# Phase 6 — Spending-pattern clusters

[`walkthrough/phase6.md`](../walkthrough/phase6.md). Clusters are fitted on one representation
only (isometric log-ratio coordinates of the core sub-composition), and *k* is chosen within
it. The chosen partition is then checked for bootstrap stability, for sensitivity to the
zero-replacement delta, and for how much of it is the pattern of which categories are zero.
These are descriptive segments, not behavioural personas.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sklearn.metrics import adjusted_rand_score

from savings_goal import config as C
from savings_goal import viz
from savings_goal.evaluation.metrics import weighted_mean
from savings_goal.features.engineer import spec_from_frame
from savings_goal.io import load_features, write_result
from savings_goal.models import personas as P

viz.use()
df = load_features()
spec = spec_from_frame(df)
CORE = spec.core_shares
before = len(df)
df = df[(df[CORE] > 0).any(axis=1)].reset_index(drop=True)
print(f"dropped {before - len(df)} households with no spend in any core category; {len(df):,} remain")
print(f"core parts: {[c.replace('_Share', '') for c in CORE]}  ->  {len(CORE) - 1} ILR coordinates")

X = P.standardise(P.ilr_coordinates(df[CORE]))

dropped 2 households with no spend in any core category; 41,516 remain
core parts: ['Groceries', 'Utilities', 'Transport', 'Healthcare', 'Clothing_Footwear', 'Miscellaneous']  ->  5 ILR coordinates


## Q2 — How many clusters, chosen within the ILR representation?

In [2]:
sweep = P.kmeans_sweep(X, range(2, 9))
print(sweep.round(4).to_string(index=False))
K = int(sweep.loc[sweep["silhouette"].idxmax(), "k"])
best_sil = float(sweep["silhouette"].max())
agree = {
    "silhouette (max)": K,
    "Davies-Bouldin (min)": int(sweep.loc[sweep["davies_bouldin"].idxmin(), "k"]),
    "Calinski-Harabasz (max)": int(sweep.loc[sweep["calinski_harabasz"].idxmax(), "k"]),
}
print(f"\nk chosen by each index: {agree}")

stability = P.bootstrap_stability(X, K, n_boot=30)
print(f"bootstrap stability at k={K}: ARI vs reference partition median {np.median(stability):.3f} "
      f"(5th-95th pct {np.quantile(stability, 0.05):.3f}-{np.quantile(stability, 0.95):.3f}, 30 resamples)")

WEAK = 0.25
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.subplots_adjust(wspace=0.35)
panels = [("silhouette", "max", "Silhouette (higher is better)"),
          ("davies_bouldin", "min", "Davies-Bouldin (lower is better)"),
          ("calinski_harabasz", "max", "Calinski-Harabasz (higher is better)")]
for ax, (col, best, title) in zip(axes, panels):
    ax.plot(sweep["k"], sweep[col], color=viz.BLUE, lw=2, zorder=2)
    viz.dot(ax, sweep["k"], sweep[col], viz.BLUE, size=6, zorder=3)
    pick = sweep.loc[sweep[col].idxmax() if best == "max" else sweep[col].idxmin()]
    viz.end_label(ax, pick["k"], pick[col], f"k={int(pick['k'])}", dy=8)
    viz.strip(ax, "y")
    ax.set_xlabel("k")
    ax.set_title(title)
axes[0].axhline(WEAK, color=viz.AXIS, lw=0.8, zorder=1)
axes[0].annotate(f"weak-structure line ({WEAK})", (sweep["k"].min(), WEAK), xytext=(0, 4),
                 textcoords="offset points", ha="left", va="bottom", color=viz.MUTED, fontsize=8)
fig.suptitle(f"All three indices pick k={K}; best silhouette {best_sil:.3f} is "
             f"{'above' if best_sil > WEAK else 'below'} the {WEAK} line", x=0.06, ha="left", y=1.04,
             color=viz.INK)
plt.savefig(C.RESULTS / "cluster_selection.png")
plt.show()

 k  silhouette  davies_bouldin  calinski_harabasz     inertia
 2      0.3705          1.4099         16154.2424 149432.1530
 3      0.4115          0.9758         17243.8291 113384.1312
 4      0.2829          1.2224         15377.0721  98320.0802
 5      0.2936          1.2857         13604.3156  89826.2429
 6      0.2358          1.3082         12765.9105  81799.0778
 7      0.2513          1.2165         12415.7521  74277.5544
 8      0.2385          1.2107         11861.5875  69185.1310

k chosen by each index: {'silhouette (max)': 3, 'Davies-Bouldin (min)': 3, 'Calinski-Harabasz (max)': 3}


bootstrap stability at k=3: ARI vs reference partition median 0.998 (5th-95th pct 0.996-0.999, 30 resamples)


## Is the partition real structure, or the zero pattern? (delta-sensitivity sweep)

In [3]:
km = P.fit_kmeans(X, K)
df["Persona"] = km.labels_
zp = P.zero_pattern(df[CORE])
ari_zero = adjusted_rand_score(zp, df["Persona"])
print(f"ARI between clusters and the raw zero-pattern of the core parts: {ari_zero:.3f}")
delta = P.delta_sensitivity(df[CORE], K, factors=(0.1, 0.25, 0.5, 0.65, 1.0))
print("\nRe-clustering with the zero-replacement delta at each multiple of the smallest positive share:")
print(delta.round(3).to_string(index=False))
print(f"\nARI vs the 0.65x baseline ranges {delta['ARI vs baseline delta'].min():.2f}-"
      f"{delta['ARI vs baseline delta'].max():.2f}; silhouette ranges {delta['silhouette'].min():.2f}-"
      f"{delta['silhouette'].max():.2f}. The partition survives the delta choice; the silhouette "
      f"that 'justifies' it is set by that arbitrary constant, and the clusters stay keyed on the "
      f"zero pattern (ARI {delta['ARI vs zero pattern'].min():.2f}-{delta['ARI vs zero pattern'].max():.2f}).")

ARI between clusters and the raw zero-pattern of the core parts: 0.858



Re-clustering with the zero-replacement delta at each multiple of the smallest positive share:
 delta factor  silhouette  ARI vs baseline delta  ARI vs zero pattern
         0.10       0.477                  0.932                0.922
         0.25       0.447                  0.955                0.899
         0.50       0.422                  0.985                0.872
         0.65       0.412                  1.000                0.858
         1.00       0.392                  0.966                0.828

ARI vs the 0.65x baseline ranges 0.93-1.00; silhouette ranges 0.39-0.48. The partition survives the delta choice; the silhouette that 'justifies' it is set by that arbitrary constant, and the clusters stay keyed on the zero pattern (ARI 0.83-0.92).


## Q1 — What do the clusters look like?

In [4]:
profile = df.groupby("Persona").agg(n=("Persona", "size"), median_income=("INCOME", "median"),
                                    median_hh_size=("Household_Size", "median"),
                                    goal_met=("Goal_Met", "mean"),
                                    median_savings_rate=("Savings_Rate", "median"))
profile["goal_met_weighted"] = df.groupby("Persona").apply(lambda g: weighted_mean(g["Goal_Met"], g["WT"]))
profile["pct_of_sample"] = (profile["n"] / len(df) * 100).round(1)
profile["pct_of_population (WT)"] = (df.groupby("Persona")["WT"].sum() / df["WT"].sum() * 100).round(1)
sig = df.groupby("Persona")[C.SHARE_COLS].mean().T
sig.index = [i.replace("_Share", "") for i in sig.index]
zero_tbl = df.groupby("Persona")[CORE].apply(lambda g: (g == 0).mean()).T
zero_tbl.index = [i.replace("_Share", "") for i in zero_tbl.index]
print(profile.round(4).to_string())
print("\nMean expense share by cluster:")
print(sig.round(3).to_string())
print("\nShare of households with ZERO spend, by cluster and core part:")
print(zero_tbl.round(3).to_string())


def describe(p):
    z = zero_tbl[p]
    absent = [i for i, v in z.items() if v > 0.5]
    return "no " + " / no ".join(a.lower() for a in absent) + " spend" if absent else "spends on all core parts"


labels = {p: describe(p) for p in profile.index}
print("\ncluster descriptions:", labels)

             n  median_income  median_hh_size  goal_met  median_savings_rate  goal_met_weighted  pct_of_sample  pct_of_population (WT)
Persona                                                                                                                               
0         4876        48820.0             4.0    0.3123              -0.1194             0.2986           11.7                    12.2
1        28937        78700.0             5.0    0.3008              -0.1452             0.2879           69.7                    71.6
2         7703        84340.0             4.0    0.3930               0.0347             0.3835           18.6                    16.2

Mean expense share by cluster:
Persona                0      1      2
Groceries          0.540  0.448  0.470
Eating_Out         0.005  0.009  0.010
Utilities          0.112  0.097  0.107
Rent               0.011  0.009  0.012
Transport          0.000  0.073  0.083
Healthcare         0.099  0.117  0.001
Education          0

## Q3 — Do the clusters relate to Goal_Met, and is that just income?

In [5]:
ct = pd.crosstab(df["Persona"], df["Goal_Met"])
chi2, p, _, _ = chi2_contingency(ct)
cramers_v = float(np.sqrt(chi2 / (ct.to_numpy().sum() * (min(ct.shape) - 1))))
spread = profile["goal_met"].max() - profile["goal_met"].min()
print(f"unconditional Goal_Met spread across clusters {spread:.3f}; chi2 {chi2:.1f}, p {p:.2g}, "
      f"Cramer's V {cramers_v:.3f}")

assoc = P.income_association(df["Persona"], df["INCOME"])
print(f"\nIs cluster membership income? NMI with income decile {assoc['NMI vs income decile']:.4f}; "
      f"Kruskal-Wallis on income H={assoc['Kruskal-Wallis H']:.1f} (p={assoc['Kruskal-Wallis p']:.2g}), "
      f"epsilon^2={assoc['epsilon squared']:.4f}")

df["Income_Decile"] = pd.qcut(df["INCOME"], 10, labels=False)
within = df.groupby(["Income_Decile", "Persona"])["Goal_Met"].mean().unstack()
print("\nGoal_Met by income decile x cluster:")
print(within.round(3).to_string())
inter = P.persona_income_interaction(df)
print(f"\nlogit Goal_Met ~ decile + cluster: LR {inter['LR persona | decile']:.1f} on {inter['df persona']} df, "
      f"p = {inter['p persona | decile']:.2g}; pseudo-R2 {inter['pseudo R2 decile only']:.4f} -> "
      f"{inter['pseudo R2 + persona']:.4f}")
print(f"decile x cluster interaction: LR {inter['LR interaction']:.1f} on {inter['df interaction']} df, "
      f"p = {inter['p interaction']:.2g}")
print("income-adjusted odds ratios vs cluster 0:",
      {k: round(v, 3) for k, v in inter["persona odds ratios (vs persona 0), income-adjusted"].items()})

unconditional Goal_Met spread across clusters 0.092; chi2 238.8, p 1.4e-52, Cramer's V 0.076

Is cluster membership income? NMI with income decile 0.0119; Kruskal-Wallis on income H=1391.3 (p=7.5e-303), epsilon^2=0.0335

Goal_Met by income decile x cluster:
Persona            0      1      2
Income_Decile                     
0              0.051  0.009  0.030
1              0.129  0.031  0.078
2              0.175  0.051  0.123
3              0.261  0.105  0.231
4              0.362  0.169  0.309
5              0.472  0.249  0.368
6              0.572  0.345  0.431
7              0.789  0.466  0.569
8              0.816  0.636  0.683
9              0.911  0.835  0.811



logit Goal_Met ~ decile + cluster: LR 774.1 on 2 df, p = 8.2e-169; pseudo-R2 0.2654 -> 0.2802
decile x cluster interaction: LR 133.5 on 18 df, p = 1.1e-19
income-adjusted odds ratios vs cluster 0: {'C(Persona)[T.1]': 0.336, 'C(Persona)[T.2]': 0.54}


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), gridspec_kw={"width_ratios": [1.1, 0.8, 1.3]})
fig.subplots_adjust(wspace=0.4)
ax = axes[0]
im = ax.imshow(sig.to_numpy(), cmap=viz.SEQUENTIAL, vmin=0, vmax=sig.to_numpy().max(), aspect="auto")
for i in range(sig.shape[0]):
    for j in range(sig.shape[1]):
        v = sig.iat[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                color="#ffffff" if v > 0.3 else viz.INK_2)
ax.set_xticks(range(sig.shape[1]), [f"C{c}" for c in sig.columns])
ax.set_yticks(range(sig.shape[0]), [i.replace("_", " ") for i in sig.index])
ax.tick_params(length=0)
for side in ("left", "bottom"):
    ax.spines[side].set_visible(False)
ax.set_title(f"Mean expense share by cluster (k={K})")

ax = axes[1]
colours = viz.CATEGORICAL[: len(profile)]
ax.bar([f"C{c}" for c in profile.index], profile["goal_met"], width=0.55, color=colours)
for i, v in enumerate(profile["goal_met"]):
    ax.annotate(f"{v:.1%}", (i, v), xytext=(0, 4), textcoords="offset points", ha="center", color=viz.INK_2)
ax.axhline(df["Goal_Met"].mean(), color=viz.AXIS, lw=0.8)
viz.strip(ax, "y")
ax.set_ylabel("share meeting the 20% benchmark")
ax.set_title(f"Unconditional: Cramer's V {cramers_v:.3f}")

ax = axes[2]
for colour, pid in zip(colours, within.columns):
    ax.plot(within.index, within[pid], color=colour, lw=2, label=f"C{pid}: {labels[pid]}")
    viz.dot(ax, within.index, within[pid], colour, size=5)
viz.strip(ax, "y")
ax.set_xticks(range(10))
ax.set_xlabel("income decile")
ax.set_ylabel("share meeting the 20% benchmark")
viz.legend(ax, loc="upper left")
ax.set_title(f"Within income deciles: cluster LR p = {inter['p persona | decile']:.1g};\n"
             f"NMI with income decile {assoc['NMI vs income decile']:.3f}")
plt.savefig(C.RESULTS / "personas.png")
plt.show()

profile.join(sig.T).to_csv(C.RESULTS / "persona_profiles.csv")
write_result("personas", {
    "k": K, "sweep": sweep, "k_by_index": agree, "bootstrap_ari": stability,
    "ari_zero_pattern": ari_zero, "delta_sensitivity": delta, "labels": {str(k): v for k, v in labels.items()},
    "cramers_v": cramers_v, "chi2_p": p, "unconditional_spread": spread, "income_association": assoc,
    "interaction": inter, "within_decile": within.reset_index(), "zero_by_cluster": zero_tbl,
})
print("wrote results/persona_profiles.csv, results/personas.json, personas.png")

wrote results/persona_profiles.csv, results/personas.json, personas.png
